# Phase 6: Model Evaluation, Operational Thresholding, and Error Analysis

**Project:** Smart Student Retention & Early Warning System  
**Evaluation Protocol:** Group-aware unseen learners ($N_{\text{test}} = 5,414$) + Chronological out-of-time evaluation (2013 -> 2014)  
**Operational Policy:** Threshold tuned strictly on validation folds (F1 maximized with $\text{precision} \ge 0.35$)  

---

## 1. Evaluation Objectives

1. **Held-Out Generalization Audit:** Compare Dummy Baseline, Logistic Regression (unweighted & balanced), and Random Forest (unweighted & balanced) on learners never observed during training.
2. **Capacity-Constrained Decision Support (Top-K):** In institutional practice, student success teams can review only limited numbers of students (e.g., $K=100, 250, 500, 1000$ or top 5-20%). We evaluate precision and recall within these operational review quotas.
3. **Probability Calibration:** Assess whether predicted probabilities correspond to true empirical withdrawal frequencies via reliability diagrams and Brier score loss.
4. **Chronological Temporal Stability:** Evaluate performance when trained exclusively on 2013 presentations (2013B & 2013J) and tested on 2014 presentations under natural operational and disjoint purged conditions.
5. **Diagnostic Error Audit:** Inspect extreme false negatives (high-risk omissions) and false positives (spurious alarms).

In [ ]:
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        if (directory / "pyproject.toml").is_file():
            return directory
    return current.parent

PROJECT_ROOT = find_project_root()
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
print(f"Project root: {PROJECT_ROOT}")
print(f"Tables dir:   {TABLES_DIR} (exists: {TABLES_DIR.exists()})")
print(f"Figures dir:  {FIGURES_DIR} (exists: {FIGURES_DIR.exists()})")

In [ ]:
# Load held-out test evaluation table
test_df = pd.read_csv(TABLES_DIR / "model_comparison_test.csv")
print("=== Held-Out Test Set Performance Across Models ===")
display(test_df[["model_key", "pr_auc", "roc_auc", "brier_score", "decision_threshold", "precision", "recall", "f1", "flagged_count", "flagged_rate"]])

In [ ]:
# 1. Decision Support Top-K Prioritization: Pooled Test Cohort
top_k_df = pd.read_csv(TABLES_DIR / 'top_k_evaluation.csv')
print('=== 1. Decision Support Top-K Prioritization: Pooled Test Cohort (Random Forest) ===')
rf_top_k = top_k_df[top_k_df['model_key'] == 'random_forest']
display(rf_top_k[['capacity_label', 'is_valid_capacity', 'precision_at_k', 'recall_at_k', 'cumulative_gain', 'positives_captured']])

# 2. Operational Per-Term Top-K Triage (Academic term cross-module caseload)
term_top_k_df = pd.read_csv(TABLES_DIR / 'top_k_evaluation_by_term.csv')
print('\n=== 2. Operational Per-Term Top-K Triage (2014 Chronological Terms, Pooling Modules) ===')
chrono_term_rf = term_top_k_df[
    (term_top_k_df['model_key'] == 'random_forest')
    & (term_top_k_df['cohort_partition'] == 'chronological_full_presentation_2014')
]
display(chrono_term_rf[['presentation_code', 'capacity_label', 'is_valid_capacity', 'precision_at_k', 'recall_at_k', 'positives_captured', 'unique_students_in_top_k', 'duplicate_student_slots', 'operational_unit', 'notes']])

# 3. Operational Per-Module-Presentation Top-K Triage (Course offering director queue)
mod_path = TABLES_DIR / 'top_k_evaluation_by_module_presentation.csv'
if mod_path.exists():
    mod_top_k_df = pd.read_csv(mod_path)
    print('\n=== 3. Operational Per-Module-Presentation Top-K Triage (Independent Offerings) ===')
    chrono_mod_rf = mod_top_k_df[
        (mod_top_k_df['model_key'] == 'random_forest')
        & (mod_top_k_df['cohort_partition'] == 'chronological_full_presentation_2014')
    ]
    display(chrono_mod_rf[['code_module', 'code_presentation', 'capacity_label', 'is_valid_capacity', 'precision_at_k', 'recall_at_k', 'positives_captured', 'notes']].head(12))


In [ ]:
# Load chronological out-of-time evaluation table
chrono_df = pd.read_csv(TABLES_DIR / "chronological_evaluation.csv")
print("=== Chronological Out-of-Time Performance (2013 to 2014) ===")
display(chrono_df[["model_key", "evaluation_design", "operational_threshold", "threshold_source", "pr_auc", "roc_auc", "precision", "recall", "f1"]])


In [ ]:
# Inspect severe false negatives (withdrew but predicted low risk)
fn_df = pd.read_csv(TABLES_DIR / "error_audit_severe_false_negatives.csv")
print("=== Top 5 Severe False Negatives (High-Risk Omissions) ===")
display(fn_df[["code_module", "code_presentation", "id_student", "predicted_risk", "vle_total_clicks_window", "vle_active_days_window", "missing_due_submissions_count"]].head(5))

# Inspect severe false positives (persisted but predicted high risk)
fp_df = pd.read_csv(TABLES_DIR / "error_audit_severe_false_positives.csv")
print("=== Top 5 Severe False Positives (Spurious Alarms) ===")
display(fp_df[["code_module", "code_presentation", "id_student", "predicted_risk", "vle_total_clicks_window", "vle_active_days_window", "missing_due_submissions_count"]].head(5))